In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (1000, 10)
Shape of y: (1000,)


In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # ❌ leakage

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print("\n--- Task 1 (Wrong) ---")
print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)


--- Task 1 (Wrong) ---
Train Accuracy: 0.87
Test Accuracy: 0.83


# Problem: Data leakage occurs because scaler is fitted on entire dataset.
# Test data information is used during training → unrealistic performance.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

In [6]:
scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("\n--- Task 2 (Correct) ---")
print("Cross-validation scores:", scores)
print("Mean Accuracy:", np.mean(scores))
print("Standard Deviation:", np.std(scores))


--- Task 2 (Correct) ---
Cross-validation scores: [0.88125 0.89375 0.875   0.84375 0.85625]
Mean Accuracy: 0.8699999999999999
Standard Deviation: 0.01785357107135714


In [7]:
depths = [1, 5, 20]

print("\n--- Task 3 (Decision Tree) ---")
print("Depth | Train Accuracy | Test Accuracy")

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    print(depth, "   |", round(train_acc, 3), "         |", round(test_acc, 3))


--- Task 3 (Decision Tree) ---
Depth | Train Accuracy | Test Accuracy
1    | 0.882          | 0.84
5    | 0.954          | 0.855
20    | 1.0          | 0.84


Depth = 1 → Underfitting (low accuracy)
Depth = 5 → Best balance (good generalization)
Depth = 20 → Overfitting (train high, test lower)

Conclusion:
Moderate depth gives best performance.